In [1]:
import sys

import torch
import transformers

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

c:\Users\ADMIN\miniconda3\envs\zfs-caption\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.12.13 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:26:47) [MSC v.1942 64 bit (AMD64)]
PyTorch: 2.11.0+cu128
Transformers: 5.15.0


In [2]:
import json
from pathlib import Path

import numpy as np
import torch

from transformers import AutoTokenizer

In [3]:
from config import (
    PROJECT_ROOT,
    DATA_ROOT,
    SPLIT_DIR,
    SUBSET_DIR,
    TOKENIZED_DIR,
    SUBSET_NAMES,
)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)
print("SPLIT_DIR    :", SPLIT_DIR)
print("SUBSET_DIR   :", SUBSET_DIR)
print("TOKENIZED_DIR:", TOKENIZED_DIR)

PROJECT_ROOT : d:\zfs-clip-image-captioning
DATA_ROOT    : d:\zfs-clip-image-captioning\data\flickr8k
SPLIT_DIR    : d:\zfs-clip-image-captioning\data\flickr8k\splits
SUBSET_DIR   : d:\zfs-clip-image-captioning\data\flickr8k\subsets
TOKENIZED_DIR: d:\zfs-clip-image-captioning\data\flickr8k\tokenized


In [4]:
# Đường dẫn tới các frozen split từ bước 1-6
split_paths = {
    "train": SPLIT_DIR / "train.json",
    "val": SPLIT_DIR / "val.json",
    "test": SPLIT_DIR / "test.json",
}

for split_name, split_path in split_paths.items():
    print(
        f"{split_name:5} | "
        f"exists = {split_path.exists()} | "
        f"path = {split_path}"
    )

train | exists = True | path = d:\zfs-clip-image-captioning\data\flickr8k\splits\train.json
val   | exists = True | path = d:\zfs-clip-image-captioning\data\flickr8k\splits\val.json
test  | exists = True | path = d:\zfs-clip-image-captioning\data\flickr8k\splits\test.json


In [5]:
#Load dữ các file test train val 
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


train_data = load_json(split_paths["train"])
val_data = load_json(split_paths["val"])
test_data = load_json(split_paths["test"])

print("Train type:", type(train_data))
print("Val type  :", type(val_data))
print("Test type :", type(test_data))

print()

print("Train size:", len(train_data))
print("Val size  :", len(val_data))
print("Test size :", len(test_data))

Train type: <class 'dict'>
Val type  : <class 'dict'>
Test type : <class 'dict'>

Train size: 6462
Val size  : 807
Test size : 809


In [6]:
#Kiểm tra 1 sample thật trong tập dữ liệu
if isinstance(train_data, dict):
    first_image_id = next(iter(train_data))
    
    print("First image_id:")
    print(first_image_id)
    
    print("\nValue:")
    print(train_data[first_image_id])

elif isinstance(train_data, list):
    print("First item:")
    print(train_data[0])

else:
    print("Unexpected data type:", type(train_data))

First image_id:
539751252_2bd88c456b.jpg

Value:
['A toddler is making a splash inside a blue paddling pool .', 'A young boy wearing blue shorts is splashing is a blue kiddie pool .', 'Child in blue trunks splashing in blue kiddie pool', 'Child in kiddie pool splashes water , sticks out tongue .', 'The child is splashing water in a small blue pool .']


In [7]:
def validate_split(split_name, data):
    # Kiểm tra toàn bộ split có đúng kiểu dictionary không
    assert isinstance(data, dict), \
        f"{split_name} must be a dictionary."

    # Những image có captions không phải list
    invalid_caption_lists = [
        image_id
        for image_id, captions in data.items()
        if not isinstance(captions, list)
    ]

    # Số caption của từng image
    caption_counts = [
        len(captions)
        for captions in data.values()
        if isinstance(captions, list)
    ]

    # Caption nào không phải string
    invalid_caption_types = [
        (image_id, caption)
        for image_id, captions in data.items()
        if isinstance(captions, list)
        for caption in captions
        if not isinstance(caption, str)
    ]

    # Caption rỗng hoặc chỉ toàn khoảng trắng
    empty_captions = [
        (image_id, caption)
        for image_id, captions in data.items()
        if isinstance(captions, list)
        for caption in captions
        if isinstance(caption, str) and not caption.strip()
    ]

    total_captions = sum(caption_counts)

    print("Images               :", len(data))
    print("Total captions       :", total_captions)
    print("Caption count/image  :", sorted(set(caption_counts)))
    print("Invalid caption lists:", len(invalid_caption_lists))
    print("Invalid caption types:", len(invalid_caption_types))
    print("Empty captions       :", len(empty_captions))
    print()

    return {
        "invalid_caption_lists": invalid_caption_lists,
        "invalid_caption_types": invalid_caption_types,
        "empty_captions": empty_captions,
    }


train_check = validate_split("train", train_data)
val_check = validate_split("val", val_data)
test_check = validate_split("test", test_data)

Images               : 6462
Total captions       : 32310
Caption count/image  : [5]
Invalid caption lists: 0
Invalid caption types: 0
Empty captions       : 0

Images               : 807
Total captions       : 4035
Caption count/image  : [5]
Invalid caption lists: 0
Invalid caption types: 0
Empty captions       : 0

Images               : 809
Total captions       : 4045
Caption count/image  : [5]
Invalid caption lists: 0
Invalid caption types: 0
Empty captions       : 0



In [8]:
def flatten_split(data):
    samples = []

    for image_id, captions in data.items():
        for caption_idx, caption in enumerate(captions):
            samples.append({
                "image_id": image_id,
                "caption_idx": caption_idx,
                "caption": caption,
            })

    return samples


train_samples = flatten_split(train_data)
val_samples = flatten_split(val_data)
test_samples = flatten_split(test_data)

print("Train samples:", len(train_samples))
print("Val samples  :", len(val_samples))
print("Test samples :", len(test_samples))

Train samples: 32310
Val samples  : 4035
Test samples : 4045


In [9]:
from config import GPT2_MODEL_NAME
print("GPT2_MODEL   :", GPT2_MODEL_NAME)

GPT2_MODEL   : openai-community/gpt2


In [10]:
# load tokenizer từ GPT2
tokenizer = AutoTokenizer.from_pretrained(
    GPT2_MODEL_NAME,
    use_fast=True,
)

print("Tokenizer class :", tokenizer.__class__.__name__)
print("Vocabulary size :", len(tokenizer))
print("Model max length:", tokenizer.model_max_length)

print("BOS token       :", tokenizer.bos_token)
print("EOS token       :", tokenizer.eos_token)
print("PAD token       :", tokenizer.pad_token)

Tokenizer class : GPT2Tokenizer
Vocabulary size : 50257
Model max length: 1024
BOS token       : <|endoftext|>
EOS token       : <|endoftext|>
PAD token       : None


In [11]:
# Thử tách token trên 1 caption
sample = train_samples[0]
sample_caption = sample["caption"]

print("Image ID   :", sample["image_id"])
print("Caption idx:", sample["caption_idx"])
print("Caption    :", sample_caption)

Image ID   : 539751252_2bd88c456b.jpg
Caption idx: 0
Caption    : A toddler is making a splash inside a blue paddling pool .


In [12]:
# Dùng tokenize cho 1 caption vừa test
encoded = tokenizer(
    sample_caption,
    add_special_tokens=False,
)

print(encoded)

{'input_ids': [32, 30773, 318, 1642, 257, 22870, 2641, 257, 4171, 14098, 1359, 5933, 764], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [13]:
# Check kỹ bên trong 1 tokenize
input_ids = encoded["input_ids"]

tokens = tokenizer.convert_ids_to_tokens(input_ids)

print("Original caption:")
print(sample_caption)

print("\nTokens:")
print(tokens)

print("\nInput IDs:")
print(input_ids)

print("\nAttention mask:")
print(encoded["attention_mask"])

print("\nDecoded:")
print(tokenizer.decode(input_ids))

Original caption:
A toddler is making a splash inside a blue paddling pool .

Tokens:
['A', 'Ġtoddler', 'Ġis', 'Ġmaking', 'Ġa', 'Ġsplash', 'Ġinside', 'Ġa', 'Ġblue', 'Ġpadd', 'ling', 'Ġpool', 'Ġ.']

Input IDs:
[32, 30773, 318, 1642, 257, 22870, 2641, 257, 4171, 14098, 1359, 5933, 764]

Attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Decoded:
A toddler is making a splash inside a blue paddling pool .


In [14]:
caption_token_lengths = []

for sample in train_samples:
    encoded = tokenizer(
        sample["caption"],
        add_special_tokens=False,
        padding=False,
        truncation=False,
    )

    caption_token_lengths.append(
        len(encoded["input_ids"])
    )

caption_token_lengths = np.array(
    caption_token_lengths
)

# Độ dài sequence thật khi train = caption + EOS
train_token_lengths = (
    caption_token_lengths + 1
)

print("Number of captions:", len(train_token_lengths))

print("\nCaption token length")
print("Min    :", caption_token_lengths.min())
print("Mean   :", caption_token_lengths.mean())
print("Median :", np.median(caption_token_lengths))
print("Max    :", caption_token_lengths.max())

print("\nTraining sequence length (caption + EOS)")
print("Min    :", train_token_lengths.min())
print("Mean   :", train_token_lengths.mean())
print("Median :", np.median(train_token_lengths))
print("Max    :", train_token_lengths.max())

Number of captions: 32310

Caption token length
Min    : 2
Mean   : 12.354627050448778
Median : 12.0
Max    : 41

Training sequence length (caption + EOS)
Min    : 3
Mean   : 13.354627050448778
Median : 13.0
Max    : 42


In [15]:
# Kiểm tra có bao nhiêu caption dài hơn 24
candidate_lengths = [24, 32, 40, 48, 64]

for max_length in candidate_lengths:
    num_over = np.sum(train_token_lengths > max_length)
    percentage = num_over / len(train_token_lengths) * 100

    print(
        f"MAX_LENGTH = {max_length:2} | "
        f"truncated = {num_over:4} captions "
        f"({percentage:.4f}%)"
    )

MAX_LENGTH = 24 | truncated =  398 captions (1.2318%)
MAX_LENGTH = 32 | truncated =   25 captions (0.0774%)
MAX_LENGTH = 40 | truncated =    1 captions (0.0031%)
MAX_LENGTH = 48 | truncated =    0 captions (0.0000%)
MAX_LENGTH = 64 | truncated =    0 captions (0.0000%)


In [16]:
long_indices = np.where(train_token_lengths > 32)[0]

print("Sequences longer than 32 tokens:", len(long_indices))
print()

for idx in long_indices:
    sample = train_samples[idx]

    print("Image ID :", sample["image_id"])
    print("Caption  :", sample["caption"])
    print("Tokens   :", train_token_lengths[idx])
    print("-" * 80)

Sequences longer than 32 tokens: 25

Image ID : 641893280_36fd6e886a.jpg
Caption  : Two brown and white dogs , one a boxer and the other a terrier , play on a rock covered hill with a blue sky and powdery clouds in the background .
Tokens   : 35
--------------------------------------------------------------------------------
Image ID : 1472249944_d887c3aeda.jpg
Caption  : A woman in an orange coat and jeans is squatting on a rock wall while a blonde woman in a red jacket stands next to her on the wall checking her electronic device
Tokens   : 35
--------------------------------------------------------------------------------
Image ID : 3108378861_d2214d971e.jpg
Caption  : A man in a white tank top , blue jeans and glasses sitting on a rock with a woman in a white tank top , blue jeans and glasses standing over him .
Tokens   : 34
--------------------------------------------------------------------------------
Image ID : 1130017585_1a219257ac.jpg
Caption  : There are three young people 

In [17]:
# GPT-2 không có PAD token mặc định
# Dùng EOS token làm PAD token

tokenizer.pad_token = tokenizer.eos_token

print("EOS token    :", tokenizer.eos_token)
print("EOS token ID :", tokenizer.eos_token_id)

print("PAD token    :", tokenizer.pad_token)
print("PAD token ID :", tokenizer.pad_token_id)

EOS token    : <|endoftext|>
EOS token ID : 50256
PAD token    : <|endoftext|>
PAD token ID : 50256


In [18]:
# Chốt độ dài token max 48
from config import GPT2_MAX_LENGTH

In [19]:
# Tokenize caption nhưng chừa 1 vị trí cho EOS
caption_ids = tokenizer.encode(
    sample_caption,
    add_special_tokens=False,
    truncation=True,
    max_length=GPT2_MAX_LENGTH - 1,
)

# Thêm EOS thủ công
input_ids = caption_ids + [tokenizer.eos_token_id]

# Mask = 1 cho caption + EOS
attention_mask = [1] * len(input_ids)

# Padding cho đủ GPT2_MAX_LENGTH
padding_length = GPT2_MAX_LENGTH - len(input_ids)

input_ids += [tokenizer.pad_token_id] * padding_length
attention_mask += [0] * padding_length

print("Input length :", len(input_ids))
print("Mask length  :", len(attention_mask))

print("\nInput IDs:")
print(input_ids)

print("\nAttention mask:")
print(attention_mask)

print("\nReal tokens   :", sum(attention_mask))
print("Padding tokens:", GPT2_MAX_LENGTH - sum(attention_mask))

Input length : 48
Mask length  : 48

Input IDs:
[32, 30773, 318, 1642, 257, 22870, 2641, 257, 4171, 14098, 1359, 5933, 764, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256]

Attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Real tokens   : 14
Padding tokens: 34


In [20]:
def tokenize_samples(samples, tokenizer, max_length):
    all_input_ids = []
    all_attention_masks = []

    for sample in samples:
        caption = sample["caption"]

        # Chừa 1 token cuối cho EOS
        caption_ids = tokenizer.encode(
            caption,
            add_special_tokens=False,
            truncation=True,
            max_length=max_length - 1,
        )

        # Caption + EOS
        input_ids = caption_ids + [
            tokenizer.eos_token_id
        ]

        # Caption và EOS đều là token thật
        attention_mask = [1] * len(input_ids)

        # Padding
        padding_length = max_length - len(input_ids)

        input_ids += [
            tokenizer.pad_token_id
        ] * padding_length

        attention_mask += [
            0
        ] * padding_length

        all_input_ids.append(input_ids)
        all_attention_masks.append(attention_mask)

    return {
        "input_ids": torch.tensor(
            all_input_ids,
            dtype=torch.long,
        ),
        "attention_mask": torch.tensor(
            all_attention_masks,
            dtype=torch.long,
        ),
    }

In [21]:
# Tokenize trên tập train
train_encoded = tokenize_samples(
    train_samples,
    tokenizer,
    GPT2_MAX_LENGTH,
)

In [22]:
# Tokenize trên tập val
val_encoded = tokenize_samples(
    val_samples,
    tokenizer,
    GPT2_MAX_LENGTH,
)

In [23]:
# Tokenize trên tập test
test_encoded = tokenize_samples(
    test_samples,
    tokenizer,
    GPT2_MAX_LENGTH,
)

In [24]:
# Check var sau khi tokenize
print("Train input_ids      :", train_encoded["input_ids"].shape)
print("Train attention_mask :", train_encoded["attention_mask"].shape)

print("Val input_ids        :", val_encoded["input_ids"].shape)
print("Val attention_mask   :", val_encoded["attention_mask"].shape)

print("Test input_ids       :", test_encoded["input_ids"].shape)
print("Test attention_mask  :", test_encoded["attention_mask"].shape)

Train input_ids      : torch.Size([32310, 48])
Train attention_mask : torch.Size([32310, 48])
Val input_ids        : torch.Size([4035, 48])
Val attention_mask   : torch.Size([4035, 48])
Test input_ids       : torch.Size([4045, 48])
Test attention_mask  : torch.Size([4045, 48])


In [25]:
def get_token_lengths(samples, tokenizer):
    captions = [
        sample["caption"]
        for sample in samples
    ]

    encoded = tokenizer(
        captions,
        add_special_tokens=False,
        padding=False,
        truncation=False,
    )

    return np.array([
        len(input_ids) + 1
        for input_ids in encoded["input_ids"]
    ])
val_token_lengths = get_token_lengths(
    val_samples,
    tokenizer,
)

test_token_lengths = get_token_lengths(
    test_samples,
    tokenizer,
)

print("VAL")
print("Max length :", val_token_lengths.max())
print(
    "Over limit :",
    np.sum(val_token_lengths > GPT2_MAX_LENGTH)
)

print("\nTEST")
print("Max length :", test_token_lengths.max())
print(
    "Over limit :",
    np.sum(test_token_lengths > GPT2_MAX_LENGTH)
)

VAL
Max length : 35
Over limit : 0

TEST
Max length : 37
Over limit : 0


In [26]:
def check_encoded(split_name, samples, encoded):
    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    print(f"===== {split_name.upper()} =====")

    print("Samples          :", len(samples))
    print("input_ids shape  :", input_ids.shape)
    print("mask shape       :", attention_mask.shape)

    print(
        "Mask values      :",
        torch.unique(attention_mask).tolist()
    )

    print(
        "Min real tokens  :",
        attention_mask.sum(dim=1).min().item()
    )

    print(
        "Max real tokens  :",
        attention_mask.sum(dim=1).max().item()
    )

    # -------------------------
    # Basic shape checks
    # -------------------------

    assert input_ids.shape == attention_mask.shape

    assert input_ids.shape[0] == len(samples)

    assert input_ids.shape[1] == GPT2_MAX_LENGTH

    # -------------------------
    # EOS check
    # -------------------------

    real_lengths = attention_mask.sum(dim=1)

    # index của token thật cuối cùng
    last_real_indices = real_lengths - 1

    row_indices = torch.arange(
        input_ids.shape[0]
    )

    last_real_tokens = input_ids[
        row_indices,
        last_real_indices
    ]

    assert torch.all(
        last_real_tokens == tokenizer.eos_token_id
    ), f"{split_name}: missing EOS"

    print("EOS check        : PASS")
    print()
check_encoded(
    "train",
    train_samples,
    train_encoded,
)

check_encoded(
    "val",
    val_samples,
    val_encoded,
)

check_encoded(
    "test",
    test_samples,
    test_encoded,
)

===== TRAIN =====
Samples          : 32310
input_ids shape  : torch.Size([32310, 48])
mask shape       : torch.Size([32310, 48])
Mask values      : [0, 1]
Min real tokens  : 3
Max real tokens  : 42
EOS check        : PASS

===== VAL =====
Samples          : 4035
input_ids shape  : torch.Size([4035, 48])
mask shape       : torch.Size([4035, 48])
Mask values      : [0, 1]
Min real tokens  : 4
Max real tokens  : 35
EOS check        : PASS

===== TEST =====
Samples          : 4045
input_ids shape  : torch.Size([4045, 48])
mask shape       : torch.Size([4045, 48])
Mask values      : [0, 1]
Min real tokens  : 3
Max real tokens  : 37
EOS check        : PASS



In [27]:
def inspect_tokenized_sample(samples, encoded, idx):
    sample = samples[idx]

    input_ids = encoded["input_ids"][idx]
    attention_mask = encoded["attention_mask"][idx]

    real_input_ids = input_ids[
        attention_mask.bool()
    ]

    # Bỏ EOS/PAD khi decode để so với caption gốc
    decoded_caption = tokenizer.decode(
        real_input_ids.tolist(),
        skip_special_tokens=True,
    )

    print("Image ID    :", sample["image_id"])
    print("Caption idx :", sample["caption_idx"])

    print("\nOriginal:")
    print(sample["caption"])

    print("\nDecoded:")
    print(decoded_caption)

    print("\nReal tokens:")
    print(attention_mask.sum().item())

    print("\nLast real token:")
    print(real_input_ids[-1].item())

    print("EOS token ID:")
    print(tokenizer.eos_token_id)

    print("\nEOS check:")
    print(
        real_input_ids[-1].item()
        == tokenizer.eos_token_id
    )

    print("\nExact caption match:")
    print(
        decoded_caption
        == sample["caption"]
    )

In [28]:
inspect_tokenized_sample(
    train_samples,
    train_encoded,
    0,
)

Image ID    : 539751252_2bd88c456b.jpg
Caption idx : 0

Original:
A toddler is making a splash inside a blue paddling pool .

Decoded:
A toddler is making a splash inside a blue paddling pool .

Real tokens:
14

Last real token:
50256
EOS token ID:
50256

EOS check:
True

Exact caption match:
True


In [29]:
inspect_tokenized_sample(
    train_samples,
    train_encoded,
    len(train_samples) // 2,
)

inspect_tokenized_sample(
    train_samples,
    train_encoded,
    len(train_samples) - 1,
)

Image ID    : 2199250692_a16b0c2ae1.jpg
Caption idx : 0

Original:
A man is pulling a cart on wheels loaded with nets and is in front of the Ocean Blue Company .

Decoded:
A man is pulling a cart on wheels loaded with nets and is in front of the Ocean Blue Company .

Real tokens:
22

Last real token:
50256
EOS token ID:
50256

EOS check:
True

Exact caption match:
True
Image ID    : 3304556387_203b9d4db0.jpg
Caption idx : 4

Original:
The man is playing in the yard with two dogs .

Decoded:
The man is playing in the yard with two dogs .

Real tokens:
12

Last real token:
50256
EOS token ID:
50256

EOS check:
True

Exact caption match:
True


In [30]:
def build_tokenized_data(
    samples,
    encoded,
    tokenizer,
    max_length,
):
    return {
        "image_ids": [
            sample["image_id"]
            for sample in samples
        ],

        "caption_indices": [
            sample["caption_idx"]
            for sample in samples
        ],

        "captions": [
            sample["caption"]
            for sample in samples
        ],

        "input_ids": encoded["input_ids"],

        "attention_mask": encoded[
            "attention_mask"
        ],

        # metadata
        "tokenizer_name": GPT2_MODEL_NAME,
        "max_length": max_length,
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.pad_token_id,
    }

In [31]:
train_tokenized = build_tokenized_data(
    train_samples,
    train_encoded,
    tokenizer,
    GPT2_MAX_LENGTH,
)

val_tokenized = build_tokenized_data(
    val_samples,
    val_encoded,
    tokenizer,
    GPT2_MAX_LENGTH,
)

test_tokenized = build_tokenized_data(
    test_samples,
    test_encoded,
    tokenizer,
    GPT2_MAX_LENGTH,
)

In [32]:
subset_tokenized = {}

for subset_name in SUBSET_NAMES:
    subset_path = (
        SUBSET_DIR / f"{subset_name}.json"
    )

    subset_data = load_json(subset_path)

    subset_samples = flatten_split(
        subset_data
    )

    subset_encoded = tokenize_samples(
        subset_samples,
        tokenizer,
        GPT2_MAX_LENGTH,
    )

    subset_tokenized[subset_name] = (
        build_tokenized_data(
            subset_samples,
            subset_encoded,
            tokenizer,
            GPT2_MAX_LENGTH,
        )
    )

    print(
        f"{subset_name:15} | "
        f"images={len(subset_data):5} | "
        f"captions={len(subset_samples):6}"
    )

train_1pct      | images=   64 | captions=   320
train_5pct      | images=  323 | captions=  1615
train_10pct     | images=  646 | captions=  3230
train_25pct     | images= 1615 | captions=  8075
train_100pct    | images= 6462 | captions= 32310


In [33]:
previous_ids = set()

for subset_name in SUBSET_NAMES:
    current_ids = set(
        subset_tokenized[subset_name]["image_ids"]
    )

    assert previous_ids.issubset(
        current_ids
    ), f"Nested subset error: {subset_name}"

    previous_ids = current_ids

print("Nested tokenized subsets: PASS")

Nested tokenized subsets: PASS


In [34]:
def check_tokenized_data(split_name, data):
    num_samples = len(data["image_ids"])

    print(f"===== {split_name.upper()} =====")

    print("image_ids       :", len(data["image_ids"]))
    print("caption_indices :", len(data["caption_indices"]))
    print("captions        :", len(data["captions"]))
    print("input_ids       :", data["input_ids"].shape)
    print("attention_mask  :", data["attention_mask"].shape)

    assert len(data["caption_indices"]) == num_samples
    assert len(data["captions"]) == num_samples
    assert data["input_ids"].shape[0] == num_samples
    assert data["attention_mask"].shape[0] == num_samples

    print("Alignment check : PASS")
    print()


check_tokenized_data(
    "train",
    train_tokenized,
)

check_tokenized_data(
    "val",
    val_tokenized,
)

check_tokenized_data(
    "test",
    test_tokenized,
)

===== TRAIN =====
image_ids       : 32310
caption_indices : 32310
captions        : 32310
input_ids       : torch.Size([32310, 48])
attention_mask  : torch.Size([32310, 48])
Alignment check : PASS

===== VAL =====
image_ids       : 4035
caption_indices : 4035
captions        : 4035
input_ids       : torch.Size([4035, 48])
attention_mask  : torch.Size([4035, 48])
Alignment check : PASS

===== TEST =====
image_ids       : 4045
caption_indices : 4045
captions        : 4045
input_ids       : torch.Size([4045, 48])
attention_mask  : torch.Size([4045, 48])
Alignment check : PASS



In [35]:
idx = 0

print("Image ID:")
print(train_tokenized["image_ids"][idx])

print("\nCaption index:")
print(train_tokenized["caption_indices"][idx])

print("\nCaption:")
print(train_tokenized["captions"][idx])

print("\nInput IDs:")
print(train_tokenized["input_ids"][idx])

print("\nAttention mask:")
print(train_tokenized["attention_mask"][idx])

Image ID:
539751252_2bd88c456b.jpg

Caption index:
0

Caption:
A toddler is making a splash inside a blue paddling pool .

Input IDs:
tensor([   32, 30773,   318,  1642,   257, 22870,  2641,   257,  4171, 14098,
         1359,  5933,   764, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256])

Attention mask:
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


In [36]:
TOKENIZED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

torch.save(
    train_tokenized,
    TOKENIZED_DIR / "train.pt",
)

torch.save(
    val_tokenized,
    TOKENIZED_DIR / "val.pt",
)

torch.save(
    test_tokenized,
    TOKENIZED_DIR / "test.pt",
)

# Save nested subsets
for subset_name, data in subset_tokenized.items():
    output_path = (
        TOKENIZED_DIR
        / f"{subset_name}.pt"
    )

    torch.save(
        data,
        output_path,
    )

    print(
        f"Saved {subset_name}: "
        f"{output_path}"
    )

print("\nSaved tokenized files to:")
print(TOKENIZED_DIR)

Saved train_1pct: d:\zfs-clip-image-captioning\data\flickr8k\tokenized\train_1pct.pt
Saved train_5pct: d:\zfs-clip-image-captioning\data\flickr8k\tokenized\train_5pct.pt
Saved train_10pct: d:\zfs-clip-image-captioning\data\flickr8k\tokenized\train_10pct.pt
Saved train_25pct: d:\zfs-clip-image-captioning\data\flickr8k\tokenized\train_25pct.pt
Saved train_100pct: d:\zfs-clip-image-captioning\data\flickr8k\tokenized\train_100pct.pt

Saved tokenized files to:
d:\zfs-clip-image-captioning\data\flickr8k\tokenized
